In [ ]:
import torch
from torch import nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import random_split
import seaborn as sns
from model.network import Network
from model.metrics import compute_model_output_metrics
import matplotlib.pyplot as plt

In [ ]:
X_train = np.load('train_x.npy')
y_train = np.load('train_y.npy')
X_test = np.load('test_x.npy')
y_test = np.load('test_y.npy')

In [ ]:
X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train).float()
X_test = torch.from_numpy(X_test)
y_test = torch.from_numpy(y_test).float()

y_train = y_train.unsqueeze(1)
y_test = y_test.unsqueeze(1)

full_train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

In [ ]:
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [ ]:
nums = X_train[:, -2].tolist()

In [ ]:
sns.histplot(x=nums, bins=50, kde=True)

In [ ]:
net = Network(emb_dim=4)

In [ ]:
net.load_state_dict(torch.load('model.pth', weights_only=True))

In [ ]:
learning_rate = 1e-3
epochs = 10
loss_fn = torch.nn.BCELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
batch_size = 32
test_loss_fn = torch.nn.BCELoss()

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device='cpu'):
    model.train()

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def test_loop(dataloader, model, loss_fn, device='cpu'):
    model.eval()
    
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.round() == y).float().sum().item() / X.size(0)

    test_loss /= len(dataloader)
    accuracy = correct / len(dataloader)

    print(f"Test Error:\n Accuracy: {100*accuracy:>0.1f}%, Avg loss: {test_loss:>8f}\n")

In [ ]:
for epoch in range(epochs):
    print(f'Epoch {epoch+1}:')
    train_loop(train_loader, net, loss_fn, optimizer)
    test_loop(val_loader, net, test_loss_fn)

In [ ]:
def describe_model(model):
    with torch.no_grad():
        probs = model(X_test)
    y_preds = probs.round().long()
    stats = compute_model_output_metrics(y_test, y_preds, probs)
    print(f'Accuracy: {round(stats['acc'] * 100, 2)}%')
    print(f'Expected Calibration Error: {round(stats['ece'] * 100, 2)}%')
    print(f'Spread: {round(stats['spread'], 4)}')
    print(f'Binary Cross Entropy Loss: {round(stats['bce'], 4)}')
    plt.figure(figsize=(8, 5))
    sns.histplot(x=probs[:, 0], kde=True, bins=20)
    plt.xlabel('Probability')
    plt.title('Estimated Win Probability Frequencies')
    plt.show()

In [ ]:
describe_model(net)

In [ ]:
torch.save(net.state_dict(), 'model.pth')